# Engine Condition Prediction — XGBoost + SMOTE with ClearML

Predicts **Engine Condition** (0 = Faulty, 1 = Good) from sensor readings.

| Step | Detail |
|------|--------|
| **Imbalance** | SMOTE applied to training fold only (via `imblearn` Pipeline) |
| **Model** | XGBoost classifier tuned with **Optuna HPO** |
| **Tracking** | ClearML — HPO curves, metrics, confusion matrix, artifact |


In [ ]:
from pathlib import Path
import os, pickle

import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from clearml import Logger, Task

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE         = 42
TEST_SIZE            = 0.2
HPO_TRIALS           = 30
CV_FOLDS             = 5
SMOTE_K_NEIGHBORS    = 5
CLEARML_PROJECT      = "605-Engine_Condition-project"
CLEARML_TASK_NAME    = "engine_condition — XGBoost + SMOTE"

In [ ]:
data_path = Path("engine_data.csv")
print(f"Dataset path: {data_path.resolve()}")

df_raw = pd.read_csv(data_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("Raw columns:", df_raw.columns.tolist())

rename_map = {
    "Engine rpm":       "engine_rpm",
    "Lub oil pressure": "lub_oil_pressure",
    "Fuel pressure":    "fuel_pressure",
    "Coolant pressure": "coolant_pressure",
    "lub oil temp":     "lub_oil_temp",
    "Coolant temp":     "coolant_temp",
    "Engine Condition": "engine_condition",
}

df = df_raw.rename(columns=rename_map).drop_duplicates().reset_index(drop=True)

TARGET_COLUMN    = "engine_condition"
NUMERIC_FEATURES = [
    "engine_rpm", "lub_oil_pressure", "fuel_pressure",
    "coolant_pressure", "lub_oil_temp", "coolant_temp",
]

missing = [c for c in NUMERIC_FEATURES + [TARGET_COLUMN] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns after rename: {missing}\nActual: {df.columns.tolist()}")

print("\nFinal columns:", df.columns.tolist())
df.head()

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

counts = df[TARGET_COLUMN].value_counts()
label_map = {0: "Faulty (0)", 1: "Good (1)"}
print("\nTarget distribution:")
display(counts.rename(label_map).to_frame("count"))

imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (majority / minority): {imbalance_ratio:.2f}x")
print(f"→ SMOTE will oversample the minority class in the training split only.")

print("\nDescriptive statistics:")
display(df[NUMERIC_FEATURES].describe().T)

In [ ]:
target_distribution = df[TARGET_COLUMN].value_counts().sort_index()
all_classes         = sorted(df[TARGET_COLUMN].unique())
target_distribution = target_distribution.reindex(all_classes, fill_value=0)

label_map  = {0: "Faulty (0)", 1: "Good (1)"}
pie_labels = [label_map.get(c, str(c)) for c in target_distribution.index]
palette    = sns.color_palette("Set2", n_colors=len(target_distribution))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Bar chart
df["_label"] = df[TARGET_COLUMN].map(label_map)
sns.countplot(data=df, x="_label", palette="Set2",
              order=[label_map[c] for c in all_classes], ax=axes[0])
axes[0].set_title("Engine Condition Distribution")
axes[0].set_xlabel("Engine Condition")
axes[0].set_ylabel("Count")
df.drop(columns=["_label"], inplace=True)

# Pie chart
target_distribution.plot(
    kind="pie", autopct="%.1f%%", ax=axes[1],
    labels=pie_labels, colors=palette,
    pctdistance=0.80, startangle=90,
)
axes[1].set_ylabel("")
axes[1].set_title("Target Share")

# Spearman correlation
corr = df[NUMERIC_FEATURES].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[2])
axes[2].set_title("Spearman Correlation (features)")

plt.tight_layout()
plt.show()

In [ ]:
X = df[NUMERIC_FEATURES].values.astype(float)
y_raw = df[TARGET_COLUMN].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = [str(c) for c in label_encoder.classes_]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train class counts before SMOTE: { {c: int((y_train==i).sum()) for i, c in enumerate(class_names)} }")

# ── Scale BEFORE SMOTE (SMOTE on scaled space) ────────────────────────────────
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Apply SMOTE to training set only ─────────────────────────────────────────
smote = SMOTE(k_neighbors=SMOTE_K_NEIGHBORS, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Train class counts  after SMOTE: { {c: int((y_train_res==i).sum()) for i, c in enumerate(class_names)} }")
print(f"Resampled train shape: {X_train_res.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before
before_counts = pd.Series(y_train).value_counts().rename(label_map)
before_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("Set2", 2),
                   edgecolor="black", rot=0)
axes[0].set_title("Class Distribution — Before SMOTE")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=11)

# After
after_counts = pd.Series(y_train_res).value_counts().rename(label_map)
after_counts.plot(kind="bar", ax=axes[1], color=sns.color_palette("Set1", 2),
                  edgecolor="black", rot=0)
axes[1].set_title("Class Distribution — After SMOTE")
axes[1].set_ylabel("Count")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=11)

plt.suptitle("SMOTE Effect on Training Set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
if not os.getenv("CLEARML_API_ACCESS_KEY") and not os.getenv("CLEARML_OFFLINE_MODE"):
    os.environ["CLEARML_OFFLINE_MODE"] = "1"
    print("ClearML credentials not detected; running in offline mode.")

task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    task_type=Task.TaskTypes.training,
    reuse_last_task_id=False,
)
logger = Logger.current_logger()

task.connect({
    "target_column":    TARGET_COLUMN,
    "features":         NUMERIC_FEATURES,
    "random_state":     RANDOM_STATE,
    "test_size":        TEST_SIZE,
    "hpo_trials":       HPO_TRIALS,
    "cv_folds":         CV_FOLDS,
    "smote_k":          SMOTE_K_NEIGHBORS,
    "classes":          class_names,
    "train_before_smote": {c: int((y_train==i).sum()) for i, c in enumerate(class_names)},
    "train_after_smote":  {c: int((y_train_res==i).sum()) for i, c in enumerate(class_names)},
})
print("ClearML task initialised.")

In [ ]:
# CV folds on the SMOTE-resampled training data
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 100, 600),
        max_depth         = trial.suggest_int("max_depth", 3, 10),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma             = trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 0.0, 2.0),
        reg_lambda        = trial.suggest_float("reg_lambda", 0.5, 3.0),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 10),
        use_label_encoder = False,
        eval_metric       = "logloss",
        random_state      = RANDOM_STATE,
        n_jobs            = -1,
    )
    model = XGBClassifier(**params)
    score = float(cross_val_score(
        model, X_train_res, y_train_res,
        cv=cv, scoring="f1_weighted", n_jobs=-1,
    ).mean())
    logger.report_scalar("hpo/cv_f1_weighted", "XGBoost", score, trial.number)
    return score

study = optuna.create_study(
    direction="maximize",
    study_name="engine_xgb_smote_hpo",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=HPO_TRIALS, show_progress_bar=True)

print(f"\nBest CV F1 (weighted): {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

logger.report_scalar("hpo/best_cv_f1_weighted", "XGBoost", study.best_value, 0)

# ── Optuna importance plot ────────────────────────────────────────────────────
importance = optuna.importance.get_param_importances(study)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(importance.keys()), list(importance.values()),
        color=sns.color_palette("viridis", len(importance)))
ax.set_title("Optuna — Hyperparameter Importances")
ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

In [ ]:
best_params = {**study.best_params,
               "use_label_encoder": False,
               "eval_metric": "logloss",
               "random_state": RANDOM_STATE,
               "n_jobs": -1}

final_model = XGBClassifier(**best_params)
final_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False,
)

y_pred = final_model.predict(X_test_scaled)

print("Final XGBoost — Classification Report")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

metrics = {
    "test_accuracy":           float(accuracy_score(y_test, y_pred)),
    "test_f1_weighted":        float(f1_score(y_test, y_pred, average="weighted")),
    "test_precision_weighted": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
    "test_recall_weighted":    float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
    "test_f1_macro":           float(f1_score(y_test, y_pred, average="macro")),
}
for k, v in metrics.items():
    logger.report_scalar(f"final/{k}", "XGBoost", v, 0)
    print(f"  {k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=class_names,
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("XGBoost — Engine Condition Confusion Matrix")

# XGBoost feature importances
importances = pd.Series(
    final_model.feature_importances_, index=NUMERIC_FEATURES
).sort_values(ascending=True)
importances.plot(kind="barh", ax=axes[1],
                 color=sns.color_palette("coolwarm", len(importances)))
axes[1].set_title("XGBoost Feature Importances (gain)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

In [ ]:
output_path = Path("..") / "artifact" / "engine_condition_xgb_smote.pkl"
output_path.parent.mkdir(parents=True, exist_ok=True)

bundle = {
    "model":            final_model,
    "scaler":           scaler,
    "label_encoder":    label_encoder,
    "target_column":    TARGET_COLUMN,
    "feature_columns":  NUMERIC_FEATURES,
    "best_params":      study.best_params,
    "metrics":          metrics,
    "smote_k":          SMOTE_K_NEIGHBORS,
}

with open(output_path, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved bundle to: {output_path.resolve()}")
task.upload_artifact("engine_xgb_smote", artifact_object=str(output_path.resolve()))
task.close()
print("Done.")

## Outcome

### Class Imbalance Handling
- **SMOTE** (`k_neighbors=5`) applied **only to the training split** — test set is never oversampled.
- Scaling (RobustScaler) is applied *before* SMOTE so synthetic points are generated in a normalised space.

### Model
- **XGBoost** tuned with **Optuna TPE sampler** (`n_trials=30`).
- Hyperparameters tuned: `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `gamma`, `reg_alpha`, `reg_lambda`, `min_child_weight`.
- CV scoring: weighted F1 on SMOTE-resampled training folds.

### ClearML Tracking
- Per-trial HPO curve, hyperparameter importance plot, final metrics, confusion matrix, feature importances, and `.pkl` artifact bundle.